In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# Vocabulary size
vocab_size = 10000

# Keep only the top 10000 most frequent words
(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words=vocab_size
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 25000
Testing samples: 25000


In [ ]:
max_length = 200
x_train=pad_sequences(X_train,maxlen=max_length,padding='post')
x_test=pad_sequences(X_test,maxlen=max_length,padding='post')

In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=32, input_length=max_length),
    SimpleRNN(64),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [ ]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history=model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 20s 104ms/step - accuracy: 0.5054 - loss: 0.6940 - val_accuracy: 0.5258 - val_loss: 0.6897
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 15s 93ms/step - accuracy: 0.5568 - loss: 0.6767 - val_accuracy: 0.5558 - val_loss: 0.6696
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 15s 94ms/step - accuracy: 0.6474 - loss: 0.5763 - val_accuracy: 0.5418 - val_loss: 0.7152
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 14s 90ms/step - accuracy: 0.7279 - loss: 0.4382 - val_accuracy: 0.5324 - val_loss: 0.8366
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 15s 93ms/step - accuracy: 0.7455 - loss: 0.3950 - val_accuracy: 0.5202 - val_loss: 0.9823


In [ ]:
loss,accuracy=model.evaluate(x_test,y_test)
print("Test Accuracy:",accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.5246 - loss: 0.9746
Test Accuracy: 0.5246000289916992


In [ ]:
from tensorflow.keras.datasets import imdb
word_index=imdb.get_word_index()
word_index={k:(v+3) for k,v in word_index.items()}
word_index["<PAD>"]=0
word_index["<START>"]=1
word_index["<UNK>"]=2
word_index["<UNUSED>"]=3

In [ ]:
def review_to_sequence(review):
  review = review.lower().split()
  sequence = []
  for word in review:
      # Get the shifted index for the word, or 2 for UNK if not found
      idx = word_index.get(word, 2)
      # If the index is too large for our vocab_size, map it to UNK (index 2)
      if idx >= vocab_size:
          sequence.append(2) # Map to UNK
      else:
          sequence.append(idx)
  return sequence

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
max_length=200
def predict_sentiment(review):
  sequence=review_to_sequence(review)
  padded_sequence=pad_sequences([sequence],maxlen=max_length,padding='post')
  prediction=model.predict(padded_sequence,verbose=0)
  score=prediction[0][0]
  if score>0.5:
    sentiment="Positive"
  else:
    sentiment="Negative"
  return sentiment,score

In [ ]:
predict_sentiment("suiiiiii")

('Negative', np.float32(0.48724085))